In [ ]:
!apt-get update
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
!nohup ollama serve >/dev/null 2>&1 &

In [ ]:
!ollama pull gemma3:4b

In [ ]:
!pip install -q ollama

In [ ]:
!pip install -q langchain-ollama

In [ ]:
# YouTube
%pip install -U youtube-search youtube-transcript-api pytube

# LangChain
%pip install -U langchain
%pip install -U langchain-core
%pip install -U langchain-community
%pip install -U langchain-text-splitters
%pip install -U langchain-huggingface
%pip install -U langchain-chroma

# Embedding
%pip install -U sentence-transformers

# Vector DB
%pip install -U chromadb

# Reranker
%pip install -U flashrank

In [ ]:
### 1. 환경 설정 하기
from youtube_search import YoutubeSearch
from langchain_community.document_loaders import YoutubeLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
### 2. Youtube 다운로드
videos = YoutubeSearch("미국 대선", max_results=5).to_dict()

docs = []
cnt = 0
for v in enumerate(videos, start=1 ):
  try:
    v["video_url"] = "https://youtube.com" + v["url_suffix"]
    loader = YoutubeLoader.from_youtube_url(
      v["video_url"],
      language=["ko", "en"]
    )
    print("비디오 자막")
  except Exception as e:
    print(e)
    continue
  docs.extend(loader.load())

In [ ]:
### 3. Splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

split_docs = splitter.split_documents(docs)
print(len(split_docs))

0


In [ ]:
### 4. HuggingFace Embedding
embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={
        "device":"cuda"
    },
    encode_kwargs={
        "normalize_embeddings":True
    }
)

In [ ]:
### 5. VectorStore
vectorstore = Chroma.from_documents(
    split_docs,
    embedding
)

In [ ]:
### 6. retriever
base_retriever = vectorstore.as_retriever(
    search_kwargs={"k":10}
)

In [ ]:
### 7. FlashRank
from langchain_classic.retrievers.document_compressors import  FlashrankRerank
reranker = FlashrankRerank(top_n=3)

In [ ]:
### 8. redundant_filter
from langchain_community.document_transformers import EmbeddingsRedundantFilter
redundant_filter = EmbeddingsRedundantFilter(embeddings=embedding)

In [ ]:
### 9. LLM
model = ChatOllama(
    model="gemma3:4b",
    temperature=0.7
)

In [ ]:
### 10. Document Compressor
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
llm_compressor = LLMChainExtractor.from_llm(model)

In [ ]:
### 11. Document Transformer
from langchain_community.document_transformers import LongContextReorder
reorder = LongContextReorder()

In [ ]:
### 12. Compressor Pipeline : 순서가 무척 중요하다.
from langchain_classic.retrievers.document_compressors import  DocumentCompressorPipeline
from langchain_classic.retrievers import ContextualCompressionRetriever
pipeline_compressor = DocumentCompressorPipeline(
    transformers=[
        redundant_filter,
        reranker,
        reorder,
        llm_compressor
    ]
)

In [ ]:
### 13. ContextualCompressionRetriever
post_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor,
    base_retriever=base_retriever
)

In [ ]:
### 14. 검색
question = "미국 대선 후보들의 경제 정책은?"
docs = post_retriever.invoke(question)

In [ ]:
### 15. context
context = "\n\n".join(
    doc.page_content
    for doc in docs
)

In [ ]:
### 16. prompt
prompt = ChatPromptTemplate.from_template(
"""
You are a helpful AI assistant.

Use ONLY the following context.

Context:
{context}

Question:
{input}

Answer:
"""
)

In [ ]:
### 17. Chain
chain = prompt | model | StrOutputParser()

In [ ]:
### 18. 실행
result = chain.invoke(
  {
      "context":context,
      "question":question
  }
)
print(result)

제공된 문서에 따르면, 미국 대선 후보들의 경제 정책은 다음과 같은 특징을 보입니다.

*   **트럼프 대통령:** 숫자(흑자)에 매우 민감하며, 당장 흑자를 줄이라는 요구를 할 가능성이 높습니다.
*   **할리 해리스 후보:** 바이든 대통령의 입장을 거의 계승할 것으로 예상됩니다.
*   **전반적인 경향:** 누가 대통령이 되든 미국 경제는 국내 기업에 부담을 가중시킬 수 있습니다.
*   **정책 방향:** 전반적으로 미국의 대외 경제 정책은 과거 트랜스파던트 시대의 자유무역 촉진에서 벗어나 보호주의 및 보호 무역을 강조하는 방향으로 전환될 것으로 예상됩니다.

이 정보는 제공된 문서에 기반하며, 실제 후보들의 구체적인 경제 정책과는 차이가 있을 수 있습니다.
